# 🧪 Fabricators AI - Testing & Development Notebook

Use this notebook to:
- Test different LLM models
- Experiment with prompts
- Measure model quality & speed
- Iterate and find the best configuration
- Save results to GitHub

**Result:** Lock best model/prompt in production config

---

## 1️⃣ Setup

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/Fabricators_ai.git
%cd Fabricators_ai

In [ ]:
# Install dependencies
!pip install -q -e .
!pip install -q unsloth_zoo transformers torch

In [ ]:
import asyncio
import logging
from pathlib import Path

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import development tools
from development.model_testing import ModelTestingFramework
from development.prompt_manager import PromptManager
from models.llm_provider import LLMProviderFactory
from services import ChatService, DesignAnalyzerService, ReportService

print("✓ Imports successful")

## 2️⃣ Scenario 1: Test Different Models

Compare multiple models to find the best one.

**What happens:**
1. Load different models (Llama-2-7b, Llama-2-13b, etc.)
2. Test each with same question
3. Compare quality and speed
4. Save results to `metrics/model_comparison.json`
5. Identify best model → Update `config/production.py`

In [ ]:
async def test_different_models():
    """Test and compare different models."""
    print("\n" + "="*60)
    print("Testing Different Models")
    print("="*60)

    framework = ModelTestingFramework()

    # Test question
    test_question = "How do I design a cube for 3D printing in plastic?"

    # Models to test
    models = [
        "meta-llama/Llama-2-7b-hf",
        "meta-llama/Llama-2-13b-hf",
        # Add more: "mistralai/Mistral-7B-v0.1"
    ]

    for model_name in models:
        try:
            logger.info(f"Testing: {model_name}")

            # Create provider
            provider = LLMProviderFactory.create(
                provider_type="unsloth",
                model_name=model_name,
                max_tokens=512,
                temperature=0.7,
            )
            await provider.initialize()

            # Test the model
            prompt = f"You are a fabrication assistant. Question: {test_question}\n\nProvide a detailed response:"
            response = await provider.generate(prompt)

            print(f"\n✓ {model_name}")
            print(f"Response: {response[:200]}...")

            # Cleanup
            await provider.shutdown()

        except Exception as e:
            logger.error(f"Error testing {model_name}: {e}")

    # Show results
    framework.print_comparison()
    framework.save_results("model_comparison.json")

# Run it
await test_different_models()

## 3️⃣ Scenario 2: Test Different Prompts

Find the best prompt for your chosen model.

**What happens:**
1. Create multiple prompt versions
2. Test same model with each prompt
3. Measure response quality
4. Save results
5. Lock best prompt → Update `instructions/fabrication_assistant.md`

In [ ]:
# First, create prompt versions to test
prompt_v1 = """
You are a helpful fabrication assistant.
Help users design parts for 3D printing.
Be concise and practical.
"""

prompt_v2 = """
You are an expert in 3D printing and fabrication.
You help users:
1. Design parts optimized for 3D printing
2. Choose the right materials
3. Estimate print time and cost

Always be specific and practical in your advice.
"""

prompt_v3 = """
You are a fabrication expert with 10 years of experience.
When users ask about 3D printing, you:
- Ask clarifying questions about their use case
- Recommend optimal materials and settings
- Warn about common mistakes
- Provide cost and time estimates
"""

prompts = {
    "v1_basic": prompt_v1,
    "v2_detailed": prompt_v2,
    "v3_expert": prompt_v3,
}

print(f"Created {len(prompts)} prompt versions to test")

In [ ]:
async def test_different_prompts():
    """Test and compare different prompts."""
    print("\n" + "="*60)
    print("Testing Different Prompts")
    print("="*60)

    framework = ModelTestingFramework()

    # Use best model from scenario 1
    model_name = "meta-llama/Llama-2-7b-hf"
    test_question = "How do I design a cube for 3D printing in plastic?"

    # Initialize model once
    provider = LLMProviderFactory.create(
        provider_type="unsloth",
        model_name=model_name,
    )
    await provider.initialize()

    # Test each prompt
    for prompt_name, prompt_template in prompts.items():
        try:
            logger.info(f"Testing prompt: {prompt_name}")

            full_prompt = prompt_template + f"\n\nUser: {test_question}\nAssistant:"
            response = await provider.generate(full_prompt)

            print(f"\n✓ {prompt_name}")
            print(f"Response: {response[:150]}...")

        except Exception as e:
            logger.error(f"Error testing {prompt_name}: {e}")

    # Cleanup
    await provider.shutdown()

    # Results
    framework.print_comparison()
    framework.save_results("prompt_comparison.json")

# Run it
await test_different_prompts()

## 4️⃣ Scenario 3: Complete System Test

Test the full system end-to-end with your best settings.

**What happens:**
1. Initialize all services
2. Run complete conversation flow
3. Generate reports
4. Verify everything works together

In [ ]:
async def test_complete_system():
    """Test the complete system end-to-end."""
    print("\n" + "="*60)
    print("Testing Complete System")
    print("="*60)

    # Initialize provider with best settings
    provider = LLMProviderFactory.create(
        provider_type="unsloth",
        model_name="meta-llama/Llama-2-7b-hf",
    )
    await provider.initialize()

    # Create services
    chat_service = ChatService(provider)
    design_analyzer = DesignAnalyzerService(provider)
    report_service = ReportService(provider, chat_service, design_analyzer)

    # Start conversation
    print("\n>>> Starting test conversation")
    session_id = await chat_service.start_conversation()
    print(f"Session: {session_id}")

    # Send messages
    messages = [
        "I want to design a cube for 3D printing",
        "What material do you recommend?",
        "How long will it take to print?",
    ]

    for msg in messages:
        print(f"\nUser: {msg}")
        response = await chat_service.send_message(session_id, msg)
        print(f"Assistant: {response.answer}")

    # Generate report
    print("\n>>> Generating report...")
    report = await report_service.generate_report(session_id)
    print(f"Report ID: {report.report_id}")
    print(f"Summary: {report.conversation_summary}")

    # Cleanup
    await provider.shutdown()
    print("\n✓ System test complete")

# Run it
await test_complete_system()

## 5️⃣ View & Save Results

In [ ]:
# Check what metrics were generated
!ls -la metrics/

In [ ]:
# Push results to GitHub
!git add metrics/
!git commit -m "Test results: model and prompt comparison"
!git push origin development

## ✅ Next Steps

After testing:

1. **Review results** in `metrics/` folder
2. **Update production config** with best model:
   - Edit `config/production.py`
   - Update `PROD_MODEL_NAME` with best model
3. **Update best prompt**:
   - Edit `instructions/fabrication_assistant.md`
   - Replace with best prompt version
4. **Commit and push** to GitHub
5. **Deploy** to production with new settings

---

**Pro tip:** Keep old test results in git. They show your iteration history!